# Amazon ML Challenge 2026: entity resolution baseline (self-contained Kaggle notebook)
Setup: **Add Input** -> attach dataset `kesavnimmagadda/amazon-ml-challenge`. **Settings -> Internet ON** (needed for pip; Kaggle requires a phone-verified account for this). No code upload needed: all source files are written by the cell below. Run cells top to bottom.

In [ ]:
# 1. write the pipeline source files
import os
files = {"src/__init__.py": "", "src/data.py": "\"\"\"TSV loading/writing helpers. Always tab-separated, never quoted.\"\"\"\nimport csv\nimport os\n\nimport pandas as pd\n\n\ndef read_source(path):\n    return pd.read_csv(path, sep=\"\\t\", dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE)\n\n\ndef read_ground_truth(path):\n    \"\"\"Return dict s1_id -> list of matched ids (empty list for singletons).\"\"\"\n    df = pd.read_csv(path, sep=\"\\t\", dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE)\n    return {a: (b.split(\",\") if b else []) for a, b in zip(df.iloc[:, 0], df.iloc[:, 1])}\n\n\ndef write_id_lists(path, s1_ids, lists, header):\n    \"\"\"lists: dict s1_id -> iterable of ids. One row per s1 id, in the given order.\"\"\"\n    os.makedirs(os.path.dirname(path) or \".\", exist_ok=True)\n    with open(path, \"w\", encoding=\"utf-8\", newline=\"\") as f:\n        f.write(\"\\t\".join(header) + \"\\n\")\n        for s in s1_ids:\n            f.write(f\"{s}\\t{','.join(lists.get(s, ()))}\\n\")\n", "src/decision.py": "\"\"\"Turning calibrated pair probabilities into per-S1 match lists.\n\nTwo modes: a global threshold, and per-S1 expected-F0.5 maximisation. Both first enforce\nthe verified one-owner rule (each S2/S3 record belongs to at most one S1 entity).\n\"\"\"\nimport numpy as np\nimport pandas as pd\n\n\ndef one_owner(df, prob_col=\"prob\", margin=0.0):\n    \"\"\"Keep, for every S2/S3 record, only its highest-probability S1 (drops ties inside ``margin``).\"\"\"\n    df = df.sort_values([\"d_pos\", prob_col], ascending=[True, False])\n    first = ~df.duplicated(\"d_pos\")\n    out = df[first]\n    if margin > 0:\n        second = df[~first].drop_duplicates(\"d_pos\").set_index(\"d_pos\")[prob_col]\n        gap = out[\"d_pos\"].map(second).fillna(0.0)\n        out = out[(out[prob_col] - gap) >= margin]\n    return out\n\n\ndef select_threshold(df, thr, prob_col=\"prob\"):\n    return df[df[prob_col] >= thr]\n\n\ndef select_expected_f05(df, prob_col=\"prob\", topk=12, draws=128, group_chunk=20_000, seed=0,\n                        miss_rate=0.0):\n    \"\"\"For each S1 pick the prefix (by prob) maximising Monte-Carlo expected F0.5.\n\n    ``miss_rate`` adds probability mass for true matches that blocking never retrieved.\n    Cutoff 0 (empty list) is scored as P(no true match).\n    \"\"\"\n    rng = np.random.default_rng(seed)\n    df = df.sort_values([\"s1_pos\", prob_col], ascending=[True, False])\n    df = df.groupby(\"s1_pos\", sort=False).head(topk)\n    s1_ids = df[\"s1_pos\"].to_numpy()\n    starts = np.r_[0, np.flatnonzero(np.diff(s1_ids)) + 1]\n    ends = np.r_[starts[1:], len(df)]\n    probs = df[prob_col].to_numpy(np.float32)\n    keep_mask = np.zeros(len(df), bool)\n    for g0 in range(0, len(starts), group_chunk):\n        st, en = starts[g0:g0 + group_chunk], ends[g0:g0 + group_chunk]\n        G = len(st)\n        P = np.zeros((G, topk), np.float32)\n        for j in range(topk):\n            has = (st + j) < en\n            P[has, j] = probs[(st + j)[has]]\n        X = rng.random((G, draws, topk), dtype=np.float32) < P[:, None, :]\n        extra = rng.random((G, draws)) < miss_rate\n        T = X.sum(2) + extra\n        cum = np.cumsum(X, axis=2)\n        ks = np.arange(1, topk + 1)\n        F = 1.25 * cum / (0.25 * T[:, :, None] + ks[None, None, :])\n        exp_f = np.concatenate([(T == 0).mean(1, keepdims=True), F.mean(1)], axis=1)\n        best_k = exp_f.argmax(1)\n        for i in range(G):\n            keep_mask[st[i]:st[i] + best_k[i]] = True\n    return df[keep_mask]\n\n\ndef to_lists(df, s1_ids, dall_ids):\n    \"\"\"DataFrame[s1_pos, d_pos] -> dict s1_id -> list of S2/S3 ids.\"\"\"\n    out = {}\n    for s, d in zip(df[\"s1_pos\"].to_numpy(), df[\"d_pos\"].to_numpy()):\n        out.setdefault(s1_ids[s], []).append(dall_ids[d])\n    return out\n", "src/features.py": "\"\"\"Pair features for the stage-1 GBDT. Deliberately country/language agnostic (no country feature).\"\"\"\nimport numpy as np\nimport pandas as pd\nfrom rapidfuzz import fuzz\nfrom rapidfuzz.distance import JaroWinkler, Levenshtein\nfrom rapidfuzz import process\n\nFUZZ = {\n    \"name_ratio\": (\"name_lat\", fuzz.ratio),\n    \"name_tset\": (\"name_lat\", fuzz.token_set_ratio),\n    \"name_tsort\": (\"name_lat\", fuzz.token_sort_ratio),\n    \"name_part\": (\"name_lat\", fuzz.partial_ratio),\n    \"core_tset\": (\"name_core\", fuzz.token_set_ratio),\n    \"name_jw\": (\"name_lat\", JaroWinkler.similarity),\n    \"addr_tset\": (\"addr_norm\", fuzz.token_set_ratio),\n    \"addr_tsort\": (\"addr_norm\", fuzz.token_sort_ratio),\n    \"addr_part\": (\"addr_norm\", fuzz.partial_ratio),\n}\n\n\ndef _pairwise(scorer, a, b):\n    try:\n        return process.cpdist(a, b, scorer=scorer, dtype=np.float32, workers=-1).astype(np.float32)\n    except AttributeError:\n        return np.array([scorer(x, y) for x, y in zip(a, b)], dtype=np.float32)\n\n\ndef _set_feats(na, nb, ca, cb):\n    \"\"\"Number-token and core-name-token features per pair (python loop; pairs are short).\"\"\"\n    out = np.zeros((len(na), 9), np.float32)\n    for i, (a, b, x, y) in enumerate(zip(na, nb, ca, cb)):\n        sa, sb = set(a.split()), set(b.split())\n        ta, tb = set(x.split()), set(y.split())\n        common = len(ta & tb)\n        out[i, 5] = float(bool(tb) and tb < ta)          # b is a truncation of a\n        out[i, 6] = float(bool(ta) and ta < tb)          # b has extra words\n        out[i, 7] = common / max(len(ta | tb), 1)\n        out[i, 8] = common\n        if not sa or not sb:\n            out[i, :5] = (0, 0, -1, 0, -1)\n            continue\n        inter = len(sa & sb)\n        big_a = [x_ for x_ in sa if len(x_) >= 2]\n        big_b = [y_ for y_ in sb if len(y_) >= 2]\n        diff = min((abs(int(p) - int(q)) for p in big_a for q in big_b), default=-1)\n        lev = min((Levenshtein.distance(p, q) for p in big_a for q in big_b), default=-1)\n        sub = any(p != q and (p.endswith(q) or q.endswith(p)) for p in sa for q in sb if len(p) > 1 and len(q) > 1)\n        out[i, :5] = (inter, inter / len(sa | sb), diff, float(sub), lev)\n    return out\n\n\ndef compute_features(cand, s1, dall, chunk=2_000_000):\n    \"\"\"cand: s1_pos, d_pos (+ retrieval columns). dall: concatenated S2+S3 frame with ``src`` column.\"\"\"\n    parts = []\n    for s in range(0, len(cand), chunk):\n        c = cand.iloc[s:s + chunk]\n        a = s1.iloc[c.s1_pos.values]\n        b = dall.iloc[c.d_pos.values]\n        f = {}\n        for name, (col, scorer) in FUZZ.items():\n            f[name] = _pairwise(scorer, a[col].tolist(), b[col].tolist())\n        nf = _set_feats(a[\"nums\"].tolist(), b[\"nums\"].tolist(), a[\"name_core\"].tolist(), b[\"name_core\"].tolist())\n        for j, n in enumerate((\"num_inter\", \"num_jac\", \"num_mindiff\", \"num_sub\", \"num_lev\",\n                               \"core_sub\", \"core_sup\", \"core_jac\", \"core_common\")):\n            f[n] = nf[:, j]\n        f[\"legal_eq\"] = (a[\"legal\"].values == b[\"legal\"].values).astype(\"int8\")\n        f[\"key_eq\"] = (a[\"name_key\"].values == b[\"name_key\"].values).astype(\"int8\")\n        f[\"len_ratio\"] = (a[\"name_lat\"].str.len().values + 1) / (b[\"name_lat\"].str.len().values + 1)\n        f[\"ntok_a\"] = a[\"name_core\"].str.count(\" \").values + 1\n        f[\"ntok_b\"] = b[\"name_core\"].str.count(\" \").values + 1\n        f[\"a_missing\"] = a[\"addr_missing\"].values\n        f[\"b_missing\"] = b[\"addr_missing\"].values\n        f[\"b_native\"] = b[\"native\"].values\n        f[\"a_native\"] = a[\"native\"].values\n        f[\"src\"] = b[\"src\"].values\n        parts.append(pd.DataFrame(f, index=c.index))\n    feats = pd.concat(parts)\n    out = pd.concat([cand, feats], axis=1)\n    return add_group_features(out)\n\n\ndef add_group_features(df):\n    \"\"\"Competition features among the S1 rows that retrieved the same S2/S3 record.\"\"\"\n    df[\"sim\"] = df[\"cos_addr\"] + df[\"cos_name\"]\n    o = df.sort_values([\"d_pos\", \"sim\"], ascending=[True, False])\n    r = o.groupby(\"d_pos\").cumcount()\n    best = o[\"sim\"].where(r == 0).groupby(o[\"d_pos\"]).transform(\"max\")\n    second = o[\"sim\"].where(r == 1).groupby(o[\"d_pos\"]).transform(\"max\").fillna(0.0)\n    df[\"rec_best\"] = best.reindex(df.index)\n    df[\"rec_n\"] = o.groupby(\"d_pos\")[\"sim\"].transform(\"size\").reindex(df.index)\n    margin = pd.Series(np.where(r == 0, o[\"sim\"] - second, o[\"sim\"] - best), index=o.index)\n    df[\"rec_margin\"] = margin.reindex(df.index)\n    df[\"s1_rank\"] = df.groupby([\"s1_pos\", \"src\"])[\"sim\"].rank(ascending=False, method=\"first\")\n    df[\"s1_best\"] = df.groupby([\"s1_pos\", \"src\"])[\"sim\"].transform(\"max\")\n    df[\"s1_gap\"] = df[\"sim\"] - df[\"s1_best\"]\n    return df\n\n\nFEATURE_COLS = (\n    [\"rank_addr\", \"rank_name\", \"rank_addrc\", \"rank_rev\", \"in_key\", \"cos_addr\", \"cos_name\", \"cos_addrc\", \"rrf\"]\n    + list(FUZZ)\n    + [\"num_inter\", \"num_jac\", \"num_mindiff\", \"num_sub\", \"num_lev\", \"core_sub\", \"core_sup\", \"core_jac\",\n       \"core_common\", \"legal_eq\", \"key_eq\", \"len_ratio\", \"ntok_a\", \"ntok_b\",\n       \"a_missing\", \"b_missing\", \"a_native\", \"b_native\", \"src\",\n       \"rec_best\", \"rec_n\", \"rec_margin\", \"s1_rank\", \"s1_best\", \"s1_gap\"]\n)\n\nSTAGE2_COLS = [\"p1\", \"s1_rank_p\", \"s1_best_p\", \"s1_sum_p\", \"s1_n_hi\", \"rec_n_hi\", \"rec_best_p\",\n               \"rec_margin_p\", \"s1_gap_p\"]\nFEATURE_COLS2 = FEATURE_COLS + STAGE2_COLS\n\n\ndef add_stage2_features(df, p1):\n    \"\"\"Competition features computed from first-stage probabilities (out-of-sample for stage-2 training rows).\"\"\"\n    df[\"p1\"] = p1.astype(np.float32)\n    df[\"s1_rank_p\"] = df.groupby([\"s1_pos\", \"src\"])[\"p1\"].rank(ascending=False, method=\"first\")\n    df[\"s1_best_p\"] = df.groupby(\"s1_pos\")[\"p1\"].transform(\"max\")\n    df[\"s1_sum_p\"] = df.groupby(\"s1_pos\")[\"p1\"].transform(\"sum\")\n    df[\"s1_gap_p\"] = df[\"p1\"] - df.groupby([\"s1_pos\", \"src\"])[\"p1\"].transform(\"max\")\n    hi = (df[\"p1\"] > 0.5).astype(\"int8\")\n    df[\"s1_n_hi\"] = hi.groupby(df[\"s1_pos\"]).transform(\"sum\")\n    df[\"rec_n_hi\"] = hi.groupby(df[\"d_pos\"]).transform(\"sum\")\n    o = df.sort_values([\"d_pos\", \"p1\"], ascending=[True, False])\n    r = o.groupby(\"d_pos\").cumcount()\n    best = o[\"p1\"].where(r == 0).groupby(o[\"d_pos\"]).transform(\"max\")\n    second = o[\"p1\"].where(r == 1).groupby(o[\"d_pos\"]).transform(\"max\").fillna(0.0)\n    df[\"rec_best_p\"] = best.reindex(df.index)\n    df[\"rec_margin_p\"] = pd.Series(np.where(r == 0, o[\"p1\"] - second, o[\"p1\"] - best), index=o.index).reindex(df.index)\n    return df\n", "src/learn_aliases.py": "\"\"\"Learn address alias pairs (NY<->new york, st<->street, mh<->maharashtra, ...) from train ground truth.\n\nAligns matched S1/S2-S3 addresses, looks at the 1-2 token leftovers on each side and keeps\nalphabetic pairs that co-occur far more often than chance. Uses only the provided data.\n\nUsage: python -m src.learn_aliases --data-dir ../../student_resource/dataset --out ../../work/aliases.json\n\"\"\"\nimport argparse\nimport json\nfrom collections import Counter\n\nimport pandas as pd\n\nfrom .data import read_ground_truth, read_source\nfrom .textnorm import norm_address\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--data-dir\", required=True)\n    ap.add_argument(\"--out\", required=True)\n    ap.add_argument(\"--n\", type=int, default=300_000)\n    ap.add_argument(\"--min-count\", type=int, default=50)\n    ap.add_argument(\"--min-purity\", type=float, default=0.4)\n    a = ap.parse_args()\n\n    tr = f\"{a.data_dir}/train\"\n    s1 = read_source(f\"{tr}/train_source1.tsv\").sample(a.n, random_state=0)\n    gt = read_ground_truth(f\"{tr}/train_ground_truth.tsv\")\n    first = {s: gt[s][0] for s in s1.entity_id if gt.get(s)}\n    need = set(first.values())\n    d = pd.concat([read_source(f\"{tr}/train_source2.tsv\"), read_source(f\"{tr}/train_source3.tsv\")])\n    d = d[d.entity_id.isin(need)].set_index(\"entity_id\")[\"business_address\"]\n    pairs, freq, side_a = Counter(), Counter(), Counter()\n    for sid, addr in zip(s1.entity_id, s1.business_address):\n        m = first.get(sid)\n        if m is None or m not in d.index or not isinstance(d[m], str):\n            continue\n        ta, tb = norm_address(addr)[0].split(), norm_address(d[m])[0].split()\n        la = [t for t in ta if t not in tb]\n        lb = [t for t in tb if t not in ta]\n        if 1 <= len(la) <= 2 and 1 <= len(lb) <= 2:\n            ka, kb = \" \".join(la), \" \".join(lb)\n            if ka.replace(\" \", \"\").isalpha() and kb.replace(\" \", \"\").isalpha():\n                pairs[(ka, kb)] += 1\n                side_a[ka] += 1\n                freq[ka] += 1\n                freq[kb] += 1\n    aliases = {}\n    for (ka, kb), n in pairs.most_common():\n        if n < a.min_count or n / side_a[ka] < a.min_purity:\n            continue\n        src, dst = (ka, kb) if freq[ka] <= freq[kb] else (kb, ka)\n        if src not in aliases and dst not in aliases:\n            aliases[src] = dst\n    with open(a.out, \"w\", encoding=\"utf-8\") as f:\n        json.dump(aliases, f, ensure_ascii=False, indent=1)\n    print(f\"learned {len(aliases)} aliases -> {a.out}\")\n    for k, v in list(aliases.items())[:30]:\n        print(f\"  {k!r} -> {v!r}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "src/metric.py": "\"\"\"Exact macro F0.5 as defined in the problem statement (singletons included).\"\"\"\n\n\ndef f05_single(pred, truth):\n    if not truth:\n        return 1.0 if not pred else 0.0\n    if not pred:\n        return 0.0\n    tp = len(pred & truth)\n    if tp == 0:\n        return 0.0\n    p, r = tp / len(pred), tp / len(truth)\n    return 1.25 * p * r / (0.25 * p + r)\n\n\ndef macro_f05(pred, truth, entities=None):\n    \"\"\"pred/truth: dict s1_id -> set of ids. Averaged over ``entities`` (default: truth keys).\"\"\"\n    entities = list(truth) if entities is None else entities\n    if not entities:\n        return 0.0\n    return sum(f05_single(set(pred.get(e, ())), set(truth.get(e, ()))) for e in entities) / len(entities)\n", "src/pipeline.py": "\"\"\"Pipeline: normalise -> multi-view retrieval -> pair features -> stage-1 GBDT -> stage-2 GBDT on\ncompetition features -> isotonic calibration -> one-owner assignment -> threshold / expected-F0.5.\n\n  python -m src.pipeline preprocess --data-dir ... --work-dir ../../work [--aliases ...]\n  python -m src.pipeline train      --data-dir ... --work-dir ../../work [--aliases ...] [--frac 0.25]\n  python -m src.pipeline predict    --data-dir ... --work-dir ../../work --out ../../output\n\"\"\"\nimport argparse\nimport json\nimport os\nimport time\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.isotonic import IsotonicRegression\n\nfrom . import textnorm\nfrom .data import read_ground_truth, read_source, write_id_lists\nfrom .decision import one_owner, select_expected_f05, select_threshold, to_lists\nfrom .features import FEATURE_COLS, FEATURE_COLS2, add_stage2_features, compute_features\nfrom .metric import macro_f05\nfrom .retrieval import gen_candidates\n\n\ndef log(msg):\n    print(f\"[{time.strftime('%H:%M:%S')}] {msg}\", flush=True)\n\n\ndef prepare(df, n_jobs):\n    return textnorm.normalize_frame(df, n_jobs=n_jobs)\n\n\ndef setup_text(work_dir, aliases):\n    \"\"\"Point the normaliser at the alias file and the vocabulary (both optional). Returns the vocab path.\"\"\"\n    if aliases:\n        os.environ[\"ER_ALIASES\"] = aliases\n        log(f\"aliases loaded: {textnorm.load_aliases(aliases)}\")\n    vp = f\"{work_dir}/vocab.json\"\n    vp = vp if os.path.exists(vp) else None\n    if vp:\n        os.environ[\"ER_VOCAB\"] = vp\n        log(f\"vocab words loaded: {textnorm.load_vocab(vp)}\")\n    return vp\n\n\ndef _prep_dir(work_dir):\n    return f\"{work_dir}/prep\"\n\n\ndef _load_prep(work_dir, split, aliases, vocab):\n    \"\"\"Return cached normalised (s1, s2, s3) for a split, or None when not preprocessed.\"\"\"\n    d = _prep_dir(work_dir)\n    files = [f\"{d}/{split}_source{i}.parquet\" for i in (1, 2, 3)]\n    if not all(os.path.exists(f) for f in files):\n        return None\n    with open(f\"{d}/meta.json\") as f:\n        meta = json.load(f)\n    if meta.get(\"aliases\") != aliases or meta.get(\"vocab\") != vocab:\n        raise SystemExit(f\"cache in {d} was built with aliases={meta.get('aliases')!r} vocab={meta.get('vocab')!r} \"\n                         f\"but this run uses aliases={aliases!r} vocab={vocab!r}; rerun `preprocess` or delete {d}\")\n    log(f\"loading cached preprocessed {split} data from {d}\")\n    return tuple(pd.read_parquet(f) for f in files)\n\n\ndef cmd_preprocess(a):\n    os.makedirs(a.work_dir, exist_ok=True)\n    vp = f\"{a.work_dir}/vocab.json\"\n    if os.path.exists(vp):\n        os.remove(vp)\n    os.environ.pop(\"ER_VOCAB\", None)\n    textnorm.load_vocab(None)\n    setup_text(a.work_dir, a.aliases)\n    names = []  # clean S1 names from train+test -> vocabulary used to split glued names (domains, hashtags)\n    for split in (\"train\", \"test\"):\n        f = f\"{a.data_dir}/{split}/{split}_source1.tsv\"\n        if os.path.exists(f):\n            names += read_source(f)[\"business_name\"].tolist()\n    t = time.time()\n    vocab = textnorm.build_vocab(names)\n    with open(vp, \"w\", encoding=\"utf-8\") as f:\n        json.dump(vocab, f, ensure_ascii=False)\n    log(f\"vocabulary: {len(vocab):,} words from {len(names):,} S1 names in {time.time() - t:.0f}s\")\n    vp = setup_text(a.work_dir, a.aliases)\n\n    out = _prep_dir(a.work_dir)\n    os.makedirs(out, exist_ok=True)\n    for split in (\"train\", \"test\"):\n        for i in (1, 2, 3):\n            src = f\"{a.data_dir}/{split}/{split}_source{i}.tsv\"\n            if not os.path.exists(src):\n                continue\n            t = time.time()\n            df = prepare(read_source(src), a.jobs)\n            df.to_parquet(f\"{out}/{split}_source{i}.parquet\", index=False)\n            log(f\"{split}_source{i}: {len(df):,} rows normalised in {time.time() - t:.0f}s\")\n            if i in (1, 2):  # small per-country sample to eyeball the cleaning\n                cols = [\"entity_id\", \"country\", \"business_name\", \"name_lat\", \"name_core\", \"legal\",\n                        \"business_address\", \"addr_norm\", \"nums\"]\n                pd.concat([g.head(300) for _, g in df.groupby(\"country\")])[cols].to_csv(\n                    f\"{out}/inspect_{split}_s{i}.tsv\", sep=\"\\t\", index=False)\n    with open(f\"{out}/meta.json\", \"w\") as f:\n        json.dump({\"aliases\": a.aliases, \"vocab\": vp}, f)\n    log(f\"done. Inspect cleaned samples: {out}/inspect_*.tsv\")\n\n\ndef concat_sources(s2, s3):\n    return pd.concat([s2.assign(src=2), s3.assign(src=3)], ignore_index=True)\n\n\ndef build_pairs(s1, dall, n_jobs, k=20, keep=15, k_addrc=10, k_rev=5):\n    \"\"\"Candidates + features, country by country (country is an open set of labels).\"\"\"\n    out = []\n    for c, s1c in s1.groupby(\"country\"):\n        parts = []\n        for src in (2, 3):\n            dc = dall[(dall.country == c) & (dall.src == src)]\n            if dc.empty:\n                continue\n            cand = gen_candidates(s1c.reset_index(drop=True), dc.reset_index(drop=True), k_addr=k, k_name=k,\n                                  k_addrc=k_addrc, k_rev=k_rev, keep=keep, n_jobs=n_jobs)\n            if cand.empty:\n                continue\n            cand[\"s1_pos\"] = s1c.index.values[cand[\"s1_pos\"].values]\n            cand[\"d_pos\"] = dc.index.values[cand[\"d_pos\"].values]\n            parts.append(cand)\n        if not parts:\n            continue\n        cand = pd.concat(parts, ignore_index=True)\n        log(f\"country={c}: {len(s1c):,} S1, {len(cand):,} candidate pairs\")\n        out.append(compute_features(cand, s1, dall))\n    return pd.concat(out, ignore_index=True)\n\n\ndef fit_model(train, val, cols):\n    try:\n        import lightgbm as lgb\n        m = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.05, num_leaves=127, subsample=0.8,\n                               subsample_freq=1, colsample_bytree=0.8, min_child_samples=50, n_jobs=-1, verbose=-1)\n        m.fit(train[cols], train[\"y\"], eval_set=[(val[cols], val[\"y\"])],\n              callbacks=[lgb.early_stopping(50, verbose=False)])\n    except Exception as e:  # e.g. missing libomp: fall back so the pipeline still runs\n        log(f\"LightGBM unavailable ({type(e).__name__}); using scikit-learn HistGradientBoosting\")\n        from sklearn.ensemble import HistGradientBoostingClassifier\n        m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=63)\n        m.fit(train[cols], train[\"y\"])\n    return m\n\n\ndef decide(df, mode, thr, s1_ids, dall_ids, margin=0.0, miss_rate=0.0):\n    own = one_owner(df, margin=margin)\n    sel = select_expected_f05(own, miss_rate=miss_rate) if mode == \"expected\" else select_threshold(own, thr)\n    return to_lists(sel, s1_ids, dall_ids)\n\n\ndef cmd_train(a):\n    vp = setup_text(a.work_dir, a.aliases)\n    tr = f\"{a.data_dir}/train\"\n    cached = _load_prep(a.work_dir, \"train\", a.aliases, vp)\n    if cached:\n        s1, d = cached[0], concat_sources(cached[1], cached[2])\n    else:\n        s1 = read_source(f\"{tr}/train_source1.tsv\")\n        d = concat_sources(read_source(f\"{tr}/train_source2.tsv\"), read_source(f\"{tr}/train_source3.tsv\"))\n    gt = read_ground_truth(f\"{tr}/train_ground_truth.tsv\")\n    owner = {m: s for s, ms in gt.items() for m in ms}\n\n    rng = np.random.default_rng(a.seed)\n    s1 = s1.sample(frac=a.frac, random_state=a.seed).reset_index(drop=True)\n    if a.limit_s1:\n        s1 = s1.iloc[:a.limit_s1]\n    matched_here = set(m for s in s1.entity_id for m in gt[s])\n    keep = d.entity_id.isin(matched_here) | (~d.entity_id.isin(owner.keys()) & (rng.random(len(d)) < a.frac))\n    d = d[keep].reset_index(drop=True)\n    log(f\"universe: {len(s1):,} S1, {len(d):,} S2/S3 records\")\n\n    if not cached:\n        s1, d = prepare(s1, a.jobs), prepare(d, a.jobs)\n    pairs = build_pairs(s1, d, a.jobs, a.k, a.keep, a.k_addrc, a.k_rev)\n    s1_ids, d_ids = s1.entity_id.values, d.entity_id.values\n    pairs[\"y\"] = (pd.Series(d_ids[pairs.d_pos.values]).map(owner).values == s1_ids[pairs.s1_pos.values]).astype(\"int8\")\n    total_true = sum(len(gt[s]) for s in s1_ids)\n    log(f\"pairs={len(pairs):,}  positives={pairs.y.sum():,}  blocking recall={pairs.y.sum() / max(total_true, 1):.4f}\")\n\n    # S1-level splits: 0 stage-1 train, 1 stage-1 early-stop, 2 stage-2 train, 3 stage-2 early-stop,\n    # 4 calibration, 5 decision tuning, 6 final held-out test\n    grp = rng.choice(7, size=len(s1), p=[0.45, 0.05, 0.2, 0.05, 0.075, 0.075, 0.1])\n    g = grp[pairs.s1_pos.values]\n    m1 = fit_model(pairs[g == 0], pairs[g == 1], FEATURE_COLS)\n    p1 = m1.predict_proba(pairs[FEATURE_COLS])[:, 1]\n    pairs = add_stage2_features(pairs, p1)\n    m2 = fit_model(pairs[g == 2], pairs[g == 3], FEATURE_COLS2)\n    p2 = m2.predict_proba(pairs[FEATURE_COLS2])[:, 1]\n    y = pairs.y.values\n\n    def score(group, mode, thr=0.0, margin=0.0):\n        ids = np.flatnonzero(grp == group)\n        sub = pairs[np.isin(pairs.s1_pos.values, ids)]\n        ents = [s1_ids[i] for i in ids]\n        return macro_f05(decide(sub, mode, thr, s1_ids, d_ids, margin), {e: set(gt[e]) for e in ents}, ents)\n\n    results = {}\n    for name, raw in ((\"stage1\", p1), (\"stage2\", p2)):\n        cal = IsotonicRegression(out_of_bounds=\"clip\").fit(raw[g == 4], y[g == 4])\n        pairs[\"prob\"] = cal.predict(raw)\n        best = (\"threshold\", 0.5, 0.0, -1.0)\n        for thr in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):\n            for mg in (0.0, 0.1, 0.2):\n                sc = score(5, \"threshold\", thr, mg)\n                if sc > best[3]:\n                    best = (\"threshold\", thr, mg, sc)\n        sc_exp = score(5, \"expected\")\n        if sc_exp > best[3]:\n            best = (\"expected\", 0.0, 0.0, sc_exp)\n        final = score(6, best[0], best[1], best[2])\n        log(f\"{name}: best={best[0]} thr={best[1]} margin={best[2]} tune={best[3]:.4f} held-out F0.5={final:.4f}\")\n        results[name] = (best, final, cal)\n    use = \"stage2\" if results[\"stage2\"][0][3] >= results[\"stage1\"][0][3] else \"stage1\"\n    best, final, cal = results[use]\n    log(f\"USING {use}: {best[0]} thr={best[1]} margin={best[2]}  held-out test F0.5={final:.4f}\")\n\n    os.makedirs(a.work_dir, exist_ok=True)\n    joblib.dump({\"m1\": m1, \"m2\": m2, \"cal\": cal, \"stage\": 2 if use == \"stage2\" else 1}, f\"{a.work_dir}/model.joblib\")\n    with open(f\"{a.work_dir}/config.json\", \"w\") as f:\n        json.dump({\"mode\": best[0], \"thr\": best[1], \"margin\": best[2], \"heldout_f05\": final, \"aliases\": a.aliases,\n                   \"vocab\": vp, \"k\": a.k, \"keep\": a.keep, \"k_addrc\": a.k_addrc, \"k_rev\": a.k_rev}, f, indent=1)\n    imp = getattr(m2 if use == \"stage2\" else m1, \"feature_importances_\", None)\n    if imp is not None:\n        cols = FEATURE_COLS2 if use == \"stage2\" else FEATURE_COLS\n        log(\"top features: \" + \", \".join(f\"{n}:{int(v)}\" for n, v in sorted(zip(cols, imp), key=lambda x: -x[1])[:12]))\n\n\ndef cmd_predict(a):\n    with open(f\"{a.work_dir}/config.json\") as f:\n        cfg = json.load(f)\n    vp = setup_text(a.work_dir, cfg.get(\"aliases\"))\n    bundle = joblib.load(f\"{a.work_dir}/model.joblib\")\n    te = f\"{a.data_dir}/test\"\n    cached = _load_prep(a.work_dir, \"test\", cfg.get(\"aliases\"), vp)\n    if cached:\n        s1, d = cached[0], concat_sources(cached[1], cached[2])\n    else:\n        s1 = read_source(f\"{te}/test_source1.tsv\")\n        d = concat_sources(read_source(f\"{te}/test_source2.tsv\"), read_source(f\"{te}/test_source3.tsv\"))\n    if a.limit_s1:\n        s1 = s1.iloc[:a.limit_s1]\n    if not cached:\n        s1, d = prepare(s1, a.jobs), prepare(d, a.jobs)\n    log(f\"test: {len(s1):,} S1, {len(d):,} S2/S3; countries={sorted(s1.country.unique())}\")\n    pairs = build_pairs(s1, d, a.jobs, cfg[\"k\"], cfg[\"keep\"], cfg[\"k_addrc\"], cfg[\"k_rev\"])\n    raw = bundle[\"m1\"].predict_proba(pairs[FEATURE_COLS])[:, 1]\n    if bundle[\"stage\"] == 2:\n        pairs = add_stage2_features(pairs, raw)\n        raw = bundle[\"m2\"].predict_proba(pairs[FEATURE_COLS2])[:, 1]\n    pairs[\"prob\"] = bundle[\"cal\"].predict(raw)\n    s1_ids, d_ids = s1.entity_id.values, d.entity_id.values\n\n    cands = to_lists(pairs, s1_ids, d_ids)\n    matches = decide(pairs, cfg[\"mode\"], cfg[\"thr\"], s1_ids, d_ids, cfg[\"margin\"])\n    write_id_lists(f\"{a.out}/candidate_pairs.tsv\", s1_ids, cands, [\"source1_entity_id\", \"candidate_entity_ids\"])\n    write_id_lists(f\"{a.out}/matching_results.tsv\", s1_ids, matches, [\"source1_entity_id\", \"matched_entity_ids\"])\n    n = sum(1 for v in matches.values() if v)\n    log(f\"wrote {a.out}: {n:,}/{len(s1_ids):,} S1 with >=1 match (train prior ~94%)\")\n    by_c = s1.assign(has=[bool(matches.get(i)) for i in s1_ids]).groupby(\"country\")[\"has\"].mean()\n    log(\"match rate by country: \" + \", \".join(f\"{c}={v:.1%}\" for c, v in by_c.items()))\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"cmd\", choices=[\"preprocess\", \"train\", \"predict\"])\n    ap.add_argument(\"--data-dir\", required=True)\n    ap.add_argument(\"--work-dir\", default=\"work\")\n    ap.add_argument(\"--out\", default=\"output\")\n    ap.add_argument(\"--aliases\", default=None)\n    ap.add_argument(\"--frac\", type=float, default=0.25)\n    ap.add_argument(\"--seed\", type=int, default=0)\n    ap.add_argument(\"--jobs\", type=int, default=8)\n    ap.add_argument(\"--limit-s1\", type=int, default=0)\n    ap.add_argument(\"--k\", type=int, default=20, help=\"top-k per view (address words / name chunks)\")\n    ap.add_argument(\"--keep\", type=int, default=15, help=\"candidates kept per S1 per source\")\n    ap.add_argument(\"--k-addrc\", type=int, default=10, help=\"top-k for the address letter-chunk view (0 = off)\")\n    ap.add_argument(\"--k-rev\", type=int, default=5, help=\"reverse retrieval S2/S3 -> S1 (0 = off)\")\n    a = ap.parse_args()\n    {\"preprocess\": cmd_preprocess, \"train\": cmd_train, \"predict\": cmd_predict}[a.cmd](a)\n\n\nif __name__ == \"__main__\":\n    main()\n", "src/retrieval.py": "\"\"\"Multi-view candidate generation: address TF-IDF, name char-n-gram TF-IDF, exact name key.\n\nFrames passed in must have a clean 0..n-1 RangeIndex; returned positions refer to those rows.\n\"\"\"\nimport numpy as np\nimport pandas as pd\nfrom joblib import Parallel, delayed\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\nRRF_K = 60\n\n\ndef _topk_chunk(Q, DT, s, e, k):\n    P = (Q[s:e] @ DT).tocsr()\n    n = e - s\n    idx = np.full((n, k), -1, np.int32)\n    sc = np.zeros((n, k), np.float32)\n    ip, ind, dat = P.indptr, P.indices, P.data\n    for r in range(n):\n        a, b = ip[r], ip[r + 1]\n        if a == b:\n            continue\n        d = dat[a:b]\n        sel = np.argpartition(-d, k - 1)[:k] if b - a > k else np.arange(b - a)\n        sel = sel[np.argsort(-d[sel])]\n        idx[r, :len(sel)] = ind[a:b][sel]\n        sc[r, :len(sel)] = d[sel]\n    return idx, sc\n\n\ndef topk_sparse(Q, D, k, chunk=500, n_jobs=-1):\n    DT = D.T.tocsr()\n    spans = [(s, min(s + chunk, Q.shape[0])) for s in range(0, Q.shape[0], chunk)]\n    res = Parallel(n_jobs=n_jobs)(delayed(_topk_chunk)(Q, DT, s, e, k) for s, e in spans)\n    return np.vstack([r[0] for r in res]), np.vstack([r[1] for r in res])\n\n\ndef pair_cos(Q, D, qi, di, chunk=1_000_000):\n    out = np.empty(len(qi), np.float32)\n    for s in range(0, len(qi), chunk):\n        e = s + chunk\n        out[s:e] = np.asarray(Q[qi[s:e]].multiply(D[di[s:e]]).sum(axis=1)).ravel()\n    return out\n\n\ndef _vec(kind, texts_d, texts_q, cap):\n    if kind == \"addr\":\n        v = TfidfVectorizer(analyzer=\"word\", ngram_range=(1, 2), token_pattern=r\"\\S+\", min_df=1,\n                            max_df=cap, sublinear_tf=True, dtype=np.float32)\n    elif kind == \"addrc\":  # letter chunks: catches glued/misspelt words such as newyork, sykesvlle\n        v = TfidfVectorizer(analyzer=\"char_wb\", ngram_range=(4, 4), min_df=2, max_df=cap,\n                            sublinear_tf=True, dtype=np.float32)\n    else:\n        v = TfidfVectorizer(analyzer=\"char_wb\", ngram_range=(3, 4), min_df=2, max_df=cap,\n                            sublinear_tf=True, dtype=np.float32)\n    D = v.fit_transform(texts_d)\n    return v.transform(texts_q), D\n\n\ndef _ranks_to_frame(idx, sc, name):\n    nq, k = idx.shape\n    q = np.repeat(np.arange(nq, dtype=np.int32), k)\n    df = pd.DataFrame({\"s1_pos\": q, \"d_pos\": idx.ravel(), f\"rank_{name}\": np.tile(np.arange(1, k + 1), nq)})\n    return df[df.d_pos >= 0]\n\n\ndef key_pairs(s1, d, cap=30):\n    sizes = d.groupby(\"name_key\").size()\n    ok = sizes[(sizes <= cap)].index\n    dd = pd.DataFrame({\"name_key\": d[\"name_key\"].values, \"d_pos\": np.arange(len(d), dtype=np.int32)})\n    dd = dd[dd.name_key.isin(ok) & (dd.name_key != \"\")]\n    ss = pd.DataFrame({\"name_key\": s1[\"name_key\"].values, \"s1_pos\": np.arange(len(s1), dtype=np.int32)})\n    m = ss.merge(dd, on=\"name_key\")[[\"s1_pos\", \"d_pos\"]]\n    m[\"in_key\"] = 1\n    return m\n\n\ndef gen_candidates(s1, d, k_addr=20, k_name=20, k_addrc=10, k_rev=5, keep=15, n_jobs=-1):\n    \"\"\"Candidates for S1 rows against one country's slice of one source.\n\n    Views: address words, name letter-chunks, address letter-chunks (k_addrc, 0 = off), exact name key,\n    and reverse retrieval (each S2/S3 record's top ``k_rev`` S1 rows by address, 0 = off).\n    Returns DataFrame[s1_pos, d_pos, rank_*, in_key, cos_*, rrf] with the ``keep`` best-fused rows per S1.\n    \"\"\"\n    if len(s1) == 0 or len(d) == 0:\n        return pd.DataFrame()\n    cap = max(50, int(0.0015 * len(d)))\n    Qa, Da = _vec(\"addr\", d[\"addr_norm\"].values, s1[\"addr_norm\"].values, cap)\n    Qn, Dn = _vec(\"name\", d[\"name_lat\"].values, s1[\"name_lat\"].values, cap)\n    views = [(\"addr\", Qa, Da, k_addr), (\"name\", Qn, Dn, k_name)]\n    if k_addrc:\n        Qc, Dc = _vec(\"addrc\", d[\"addr_norm\"].values, s1[\"addr_norm\"].values, cap)\n        views.append((\"addrc\", Qc, Dc, k_addrc))\n    frames = []\n    for name, Q, D, k in views:\n        idx, sc = topk_sparse(Q, D, min(k, D.shape[0]), n_jobs=n_jobs)\n        frames.append(_ranks_to_frame(idx, sc, name))\n    if k_rev:\n        idx, sc = topk_sparse(Da, Qa, min(k_rev, Qa.shape[0]), n_jobs=n_jobs)\n        frames.append(_ranks_to_frame(idx, sc, \"rev\").rename(columns={\"s1_pos\": \"d_pos\", \"d_pos\": \"s1_pos\"}))\n    frames.append(key_pairs(s1, d))\n    cand = frames[0]\n    for f in frames[1:]:\n        cand = cand.merge(f, on=[\"s1_pos\", \"d_pos\"], how=\"outer\")\n    cand[\"in_key\"] = cand[\"in_key\"].fillna(0).astype(\"int8\")\n    for col in (\"rank_addr\", \"rank_name\", \"rank_addrc\", \"rank_rev\"):\n        if col not in cand:\n            cand[col] = np.nan\n    cand[\"rrf\"] = 0.01 * cand[\"in_key\"]\n    for col in (\"rank_addr\", \"rank_name\", \"rank_addrc\", \"rank_rev\"):\n        cand[\"rrf\"] += 1 / (RRF_K + cand[col].fillna(1e6))\n    cand = cand.sort_values([\"s1_pos\", \"rrf\"], ascending=[True, False])\n    cand = cand.groupby(\"s1_pos\", sort=False).head(keep).reset_index(drop=True)\n    qi, di = cand.s1_pos.values, cand.d_pos.values\n    cand[\"cos_addr\"] = pair_cos(Qa, Da, qi, di)\n    cand[\"cos_name\"] = pair_cos(Qn, Dn, qi, di)\n    cand[\"cos_addrc\"] = pair_cos(Qc, Dc, qi, di) if k_addrc else np.float32(np.nan)\n    return cand\n", "src/textnorm.py": "\"\"\"Language-agnostic normalisation of business names and addresses.\n\nEverything works on a Latin-folded (unidecode) view so the same code runs for US,\nIndia and France. Abbreviation tables are a small static seed that can be extended\nwith data-driven aliases from ``learn_aliases.py`` (loaded from the path in the\n``ER_ALIASES`` environment variable).\n\"\"\"\nimport json\nimport math\nimport os\nimport re\nimport unicodedata\nfrom concurrent.futures import ProcessPoolExecutor\n\nimport pandas as pd\nfrom unidecode import unidecode\n\nLEGAL_CANON = {\n    \"inc\": \"inc\", \"incorporated\": \"inc\", \"lnc\": \"inc\", \"incorp\": \"inc\",\n    \"llc\": \"llc\", \"llp\": \"llp\", \"lp\": \"lp\", \"plc\": \"plc\", \"gmbh\": \"gmbh\",\n    \"ltd\": \"ltd\", \"limited\": \"ltd\", \"lted\": \"ltd\",\n    \"pvt\": \"pvt\", \"private\": \"pvt\", \"privaet\": \"pvt\", \"prvt\": \"pvt\",\n    \"corp\": \"corp\", \"corporation\": \"corp\", \"co\": \"co\", \"company\": \"co\",\n    \"sarl\": \"sarl\", \"sas\": \"sas\", \"sasu\": \"sasu\", \"eurl\": \"eurl\", \"sci\": \"sci\",\n    \"sa\": \"sa\", \"snc\": \"snc\", \"scop\": \"scop\",\n}\nNAME_STOP = {\"and\", \"the\", \"of\", \"de\", \"la\", \"le\", \"les\", \"du\", \"des\", \"et\"}\n\nADDR_CANON = {\n    \"street\": \"st\", \"road\": \"rd\", \"avenue\": \"ave\", \"av\": \"ave\", \"boulevard\": \"blvd\",\n    \"bd\": \"blvd\", \"drive\": \"dr\", \"lane\": \"ln\", \"court\": \"ct\", \"circle\": \"cir\",\n    \"highway\": \"hwy\", \"parkway\": \"pkwy\", \"place\": \"pl\", \"terrace\": \"ter\",\n    \"r\": \"rue\", \"square\": \"sq\", \"suite\": \"ste\",\n    \"ch\": \"chemin\", \"all\": \"allee\", \"imp\": \"impasse\", \"rte\": \"route\", \"fbg\": \"faubourg\",\n}\nADDR_DROP = {\n    \"unit\", \"ste\", \"apt\", \"flr\", \"floor\", \"room\", \"no\", \"number\", \"nr\", \"near\",\n    \"opp\", \"opposite\", \"behind\", \"h\", \"hn\", \"hno\", \"door\", \"plot\", \"null\", \"nan\",\n    \"none\", \"and\",\n}\nORDINAL_SUFFIX = {\"nd\", \"rd\", \"th\"}\n\n_DOT3 = re.compile(r\"\\b([a-z])\\.([a-z])\\.([a-z])\\.?\")\n_TLD = re.compile(r\"\\.(com|net|org|in|fr|io|co\\.in|co\\.uk)\\b\")\n_STORE = re.compile(r\"#\\s*\\d+\")\n_ALNUM = re.compile(r\"[a-z0-9]+\")\n_SPLIT = re.compile(r\"[a-z]+|\\d+\")\n_LEET = str.maketrans(\"013457\", \"oleass\")\n\n_HANDLE = re.compile(r\"(^|\\s)[#@]\\w|www\\.\")\n\n_ALIASES = {}\n_VOCAB = {}\n_VTOTAL = 1.0\n\n\ndef load_aliases(path):\n    global _ALIASES\n    if path and os.path.exists(path):\n        with open(path, encoding=\"utf-8\") as f:\n            _ALIASES = json.load(f)\n    return len(_ALIASES)\n\n\nload_aliases(os.environ.get(\"ER_ALIASES\"))\n\n\ndef load_vocab(path):\n    \"\"\"Word counts from clean S1 names; used only to split glued names like suzygillenpeak.com.\"\"\"\n    global _VOCAB, _VTOTAL\n    if path and os.path.exists(path):\n        with open(path, encoding=\"utf-8\") as f:\n            _VOCAB = json.load(f)\n        _VTOTAL = float(sum(_VOCAB.values())) + 1.0\n    return len(_VOCAB)\n\n\nload_vocab(os.environ.get(\"ER_VOCAB\"))\n\n\ndef segment(tok, maxlen=20):\n    \"\"\"Viterbi word segmentation over known vocabulary words only; returns [tok] if not splittable.\"\"\"\n    if not _VOCAB or len(tok) < 8:\n        return [tok]\n    n = len(tok)\n    best, back = [0.0] + [math.inf] * n, [0] * (n + 1)\n    for i in range(1, n + 1):\n        for j in range(max(0, i - maxlen), i):\n            c = _VOCAB.get(tok[j:i])\n            if c is None or best[j] == math.inf or i - j < 3:  # pieces of 1-2 letters cause bad splits\n                continue\n            cost = best[j] - math.log(c / _VTOTAL) + 3.0  # per-word penalty: prefer fewer words\n            if cost < best[i]:\n                best[i], back[i] = cost, j\n    if best[n] == math.inf:\n        return [tok]\n    out, i = [], n\n    while i > 0:\n        out.append(tok[back[i]:i])\n        i = back[i]\n    out.reverse()\n    return out if len(out) > 1 else [tok]\n\n\ndef build_vocab(names, min_count=3):\n    from collections import Counter\n    c = Counter()\n    for n in names:\n        for t in norm_name(n)[0].split():\n            if t.isalpha() and len(t) >= 2:\n                c[t] += 1\n    return {w: k for w, k in c.items() if k >= min_count}\n\n\ndef _fold(s):\n    return unidecode(unicodedata.normalize(\"NFKC\", s)).lower()\n\n\ndef _has_native(s):\n    return any(ord(c) > 0x24F for c in s)\n\n\ndef norm_name(raw):\n    \"\"\"Return (name_lat, name_core, name_key, legal, native).\"\"\"\n    s = raw if isinstance(raw, str) else \"\"\n    native = _has_native(s)\n    s = _fold(s).replace(\"&\", \" and \")\n    s = _DOT3.sub(r\"\\1\\2\\3\", s)\n    glued = bool(_TLD.search(s) or _HANDLE.search(s))  # domain / handle / hashtag names lose their spaces\n    s = _TLD.sub(\" \", s)\n    s = _STORE.sub(\" \", s)\n    toks = []\n    for t in _ALNUM.findall(s):\n        if t == \"www\":\n            continue\n        if t.isdigit():\n            if len(t) >= 7:  # phone number\n                continue\n            toks.append(t)\n        elif t.isalpha():\n            toks.append(t)\n        else:\n            letters = sum(c.isalpha() for c in t)\n            digits = len(t) - letters\n            if letters >= 3 and digits <= 2:  # leetspeak: transp0rt\n                toks.append(t.translate(_LEET))\n            else:\n                toks.extend(_SPLIT.findall(t))\n    if glued and _VOCAB:\n        exp = []\n        for t in toks:\n            exp.extend(segment(t) if t.isalpha() and len(t) >= 8 and t not in _VOCAB else [t])\n        toks = exp\n    if toks and toks[0] == \"the\":\n        toks = toks[1:]\n    legal = sorted({LEGAL_CANON[t] for t in toks if t in LEGAL_CANON})\n    core = [t for t in toks if t not in LEGAL_CANON] or toks\n    key = sorted(t for t in core if t not in NAME_STOP) or sorted(core)\n    return \" \".join(toks), \" \".join(core), \" \".join(key), \" \".join(legal), native\n\n\ndef norm_address(raw):\n    \"\"\"Return (addr_norm, nums).\"\"\"\n    s = raw if isinstance(raw, str) else \"\"\n    toks = _SPLIT.findall(_fold(s))\n    out = []\n    for t in toks:\n        if t in ORDINAL_SUFFIX and out and out[-1].isdigit():\n            continue\n        out.append(t)\n    if _ALIASES:\n        merged, i = [], 0\n        while i < len(out):\n            bi = \" \".join(out[i:i + 2])\n            if i + 1 < len(out) and bi in _ALIASES:\n                merged.append(_ALIASES[bi])\n                i += 2\n            else:\n                merged.append(_ALIASES.get(out[i], out[i]))\n                i += 1\n        out = merged\n    out = [ADDR_CANON.get(t, t) for t in out]\n    out = [t for t in out if t not in ADDR_DROP]\n    nums = [t for t in out if t.isdigit()]\n    return \" \".join(out), \" \".join(nums)\n\n\ndef _chunk(args):\n    names, addrs = args\n    return [norm_name(n) + norm_address(a) for n, a in zip(names, addrs)]\n\n\ndef normalize_frame(df, n_jobs=8, chunk=200_000):\n    \"\"\"Add normalised columns to a source frame. Row order is preserved.\"\"\"\n    names, addrs = df[\"business_name\"].tolist(), df[\"business_address\"].tolist()\n    parts = [(names[i:i + chunk], addrs[i:i + chunk]) for i in range(0, len(df), chunk)]\n    if n_jobs > 1 and len(parts) > 1:\n        with ProcessPoolExecutor(n_jobs) as ex:\n            res = [r for part in ex.map(_chunk, parts) for r in part]\n    else:\n        res = [r for p in parts for r in _chunk(p)]\n    cols = [\"name_lat\", \"name_core\", \"name_key\", \"legal\", \"native\", \"addr_norm\", \"nums\"]\n    out = pd.DataFrame(res, columns=cols, index=df.index)\n    out[\"addr_missing\"] = (out[\"addr_norm\"] == \"\").astype(\"int8\")\n    out[\"native\"] = out[\"native\"].astype(\"int8\")\n    return pd.concat([df.reset_index(drop=True), out.reset_index(drop=True)], axis=1)\n", "tools/validate_submission.py": "#!/usr/bin/env python3\n\"\"\"\nML Challenge 2026 — Submission Validator\n\nRun this BEFORE submitting. It checks your output files against every formatting\nrule the scorer enforces, so you can catch a rejection locally instead of burning\na submission. It reads only your output files and the test source files (to learn\nwhich S1 entities are required and which S2/S3 IDs exist); it never needs the\nground truth and never computes your score.\n\nIt validates two files:\n\n* ``matching_results.tsv`` (required) — your final matches, the file scored on the\n  leaderboard.\n* ``candidate_pairs.tsv`` (optional) — the candidate set from your blocking stage.\n  When present, the validator also checks that your final matches are a subset of\n  your candidates and *warns* (never fails) otherwise. When absent it is skipped\n  with a warning; it is still expected in your final submission zip.\n\nStdlib only, Python 3.8+. Run from the ``student_resource/`` directory::\n\n    python3 utils/validate_submission.py \\\n        --matching output/matching_results.tsv \\\n        --candidate output/candidate_pairs.tsv \\\n        --test-dir dataset/test\n\nExit code 0 means the files are safe to submit; 1 means fix the listed issues\n(warnings never fail the run).\n\nID-existence check (off by default). By default the validator does NOT check that\nevery matched/candidate ID actually exists in the test set: that check loads all\nSource-2/3 IDs into memory, which on the full ~1.7M-entity test set costs a few GB\n(more when ``candidate_pairs.tsv`` is included). The default run therefore stays fast\nand light and verifies every other rule; it prints a warning noting the check was\nskipped. Pass ``--check-ids`` to turn it on (it reads ``test_source2.tsv`` /\n``test_source3.tsv`` from ``--test-dir``); a missing/garbage matched ID only lowers\nyour score rather than being rejected by the scorer, so this check is a diagnostic,\nnot a gate. If ``--check-ids`` runs out of memory, drop ``--candidate`` (the candidate\ncross-check is the biggest memory user, and the matching file is the only one scored).\n\"\"\"\n\nimport argparse\nimport os\nimport sys\n\nDELIM = \"\\t\"\nMAX_EXAMPLES = 5  # how many offending IDs to show per issue\nMATCHING_HEADER = [\"source1_entity_id\", \"matched_entity_ids\"]\nCANDIDATE_HEADER = [\"source1_entity_id\", \"candidate_entity_ids\"]\n\n\ndef read_ids(path):\n    \"\"\"Return the set of first-column entity IDs from a source TSV.\n\n    The header row is skipped and blank lines are ignored.\n    \"\"\"\n    with open(path, encoding=\"utf-8\") as f:\n        next(f, None)  # skip header\n        return {line.split(DELIM, 1)[0].strip() for line in f if line.strip()}\n\n\ndef examples(items):\n    \"\"\"Return a short, human-readable sample of ``items`` for an error message.\"\"\"\n    items = sorted(items)\n    shown = \", \".join(items[:MAX_EXAMPLES])\n    if len(items) > MAX_EXAMPLES:\n        return f\"{len(items)} total, e.g. {shown}, ...\"\n    return shown\n\n\ndef load_match_targets(test_dir, warnings):\n    \"\"\"Return the set of valid S2/S3 match IDs, or ``None`` if unavailable.\n\n    Only called when ``--check-ids`` is on. When ``test_source2.tsv`` or\n    ``test_source3.tsv`` is missing we cannot check that matched IDs exist, so we\n    record a warning and return ``None`` to signal that the existence check should be\n    skipped.\n    \"\"\"\n    targets = set()\n    for name in (\"test_source2.tsv\", \"test_source3.tsv\"):\n        path = os.path.join(test_dir, name)\n        if not os.path.isfile(path):\n            warnings.append(\n                f\"{path} not found — skipping the (optional) check that matched \"\n                f\"IDs exist in the test set. Every other rule is still checked. \"\n                f\"This is the lighter-memory mode; provide test_source2/3.tsv to \"\n                f\"enable the ID-existence check.\"\n            )\n            return None\n        targets |= read_ids(path)\n    return targets\n\n\ndef validate_id_list_file(path, expected_header, col_label, required, valid_ids, errors):\n    \"\"\"Validate one results-style TSV (matching or candidate).\n\n    Applies the shared formatting rules and appends any problems to ``errors``.\n    Returns a ``{source1_id: set(matched/candidate ids)}`` mapping, or ``None`` on a\n    fatal problem (missing file, empty file, or a broken header) that stops parsing.\n    \"\"\"\n    if not os.path.isfile(path):\n        errors.append(f\"File not found: {path}\")\n        return None\n\n    name = os.path.basename(path)\n    mapping = {}\n    seen, dup_rows, intra_dupes = set(), set(), set()\n    self_matches, wrong_prefix, unknown = set(), set(), set()\n    n_rows = empties = 0\n\n    with open(path, encoding=\"utf-8\") as f:\n        header = f.readline()\n        if not header:\n            errors.append(f\"{name} is empty.\")\n            return None\n        if DELIM not in header and \",\" in header:  # the #1 mistake: a CSV\n            errors.append(\n                f\"{name}: header has no TAB but contains commas — the file looks \"\n                \"COMMA-separated. Submissions must be TAB-separated (.tsv); \"\n                \"write it with df.to_csv(sep='\\\\t', index=False).\"\n            )\n            return None\n        cols = [c.strip().lower() for c in header.rstrip(\"\\n\").split(DELIM)]\n        if cols != expected_header:\n            errors.append(\n                f\"{name}: unexpected header {cols}. \"\n                f\"Expected exactly {expected_header} (tab-separated).\"\n            )\n            return None\n\n        for line_num, line in enumerate(f, start=2):\n            s1, tab, rest = line.partition(DELIM)\n            if not tab:\n                if s1.strip():\n                    errors.append(\n                        f\"{name}: malformed row (no tab) at line {line_num}: \"\n                        f\"{line.rstrip()!r}\"\n                    )\n                continue\n\n            n_rows += 1\n            if s1 in seen:\n                dup_rows.add(s1)\n            seen.add(s1)\n\n            ids = rest.rstrip(\"\\n\").split(\",\") if rest.strip() else []\n            if not ids:\n                empties += 1\n                mapping[s1] = set()\n                continue\n            if len(ids) != len(set(ids)):\n                intra_dupes.add(s1)\n            id_set = set(ids)\n            mapping[s1] = id_set\n            for mid in id_set:\n                if mid.startswith(\"S1-\"):\n                    self_matches.add(mid)\n                elif not mid.startswith((\"S2-\", \"S3-\")):\n                    wrong_prefix.add(mid)\n                elif valid_ids is not None and mid not in valid_ids:\n                    unknown.add(mid)\n\n    # Aggregate the per-category findings. Each entry is (offenders, message);\n    # only non-empty categories become errors.\n    findings = [\n        (\n            dup_rows,\n            \"{name}: duplicate source1_entity_id row(s): {ex}. \"\n            \"Each S1 entity may appear on only one row.\",\n        ),\n        (\n            intra_dupes,\n            \"{name}: repeated ID inside a {col} list for: {ex}. \"\n            \"No duplicate IDs are allowed within a list.\",\n        ),\n        (\n            self_matches,\n            \"{name}: {col} contains Source-1 IDs (self-matches): {ex}. \"\n            \"Only S2-/S3- IDs are allowed.\",\n        ),\n        (\n            wrong_prefix,\n            \"{name}: {col} contains IDs without an S2-/S3- prefix: {ex}.\",\n        ),\n        (\n            unknown,\n            \"{name}: {col} references IDs not in the test \"\n            \"Source-2/3 files: {ex}.\",\n        ),\n        (\n            required - seen,\n            \"{name}: required S1 entity(ies) missing: {ex}. \"\n            \"Every entity in test_source1.tsv needs a row (empty = no match).\",\n        ),\n        (\n            seen - required,\n            \"{name}: row(s) using an S1 ID that is not in the test set: {ex}.\",\n        ),\n    ]\n    for offenders, message in findings:\n        if offenders:\n            errors.append(message.format(name=name, ex=examples(offenders), col=col_label))\n\n    print(f\"  {name}: {n_rows} rows ({empties} empty, {n_rows - empties} non-empty).\")\n    return mapping\n\n\ndef validate(matching_path, candidate_path, test_dir, check_ids=False):\n    \"\"\"Validate the submission output(s); return ``(errors, warnings)`` lists.\n\n    ``check_ids`` (``--check-ids``) turns on the optional, memory-heavy check that\n    every matched/candidate ID exists in the test Source-2/3 files. It is off by\n    default so the common run stays fast and light.\n    \"\"\"\n    errors, warnings = [], []\n\n    source1 = os.path.join(test_dir, \"test_source1.tsv\")\n    if not os.path.isfile(source1):\n        errors.append(f\"Test source1 file not found: {source1} (check --test-dir).\")\n        return errors, warnings\n    required = read_ids(source1)\n    print(f\"  required S1 entities: {len(required)}\")\n\n    if check_ids:\n        valid_ids = load_match_targets(test_dir, warnings)\n        if valid_ids is not None:\n            print(f\"  valid S2/S3 match IDs: {len(valid_ids)}\")\n    else:\n        valid_ids = None\n        warnings.append(\n            \"ID-existence check is OFF (the default) — not checking that matched/\"\n            \"candidate IDs exist in the test set. Every other rule is still checked. \"\n            \"Re-run with --check-ids to enable it (needs test_source2/3.tsv; uses \"\n            \"more memory). A nonexistent ID only lowers your score, never rejects \"\n            \"your submission.\"\n        )\n\n    matched = validate_id_list_file(\n        matching_path, MATCHING_HEADER, \"matched_entity_ids\", required, valid_ids, errors\n    )\n\n    # candidate_pairs.tsv is optional: if it's absent we skip its checks with a\n    # warning (it's still expected in your final submission zip). A missing\n    # candidate file never fails this run on its own.\n    candidate = None\n    if candidate_path and os.path.isfile(candidate_path):\n        candidate = validate_id_list_file(\n            candidate_path, CANDIDATE_HEADER, \"candidate_entity_ids\",\n            required, valid_ids, errors,\n        )\n    elif candidate_path:\n        warnings.append(\n            f\"{candidate_path} not found — skipping candidate_pairs.tsv checks. \"\n            \"It is optional here, but your final submission zip must include \"\n            \"output/candidate_pairs.tsv.\"\n        )\n\n    # Soft check: your final matches should come from your blocking candidates.\n    # A matched ID absent from candidate_pairs.tsv usually means a pipeline bug,\n    # so we warn but never fail on it.\n    if matched is not None and candidate is not None:\n        offenders = {\n            s1 for s1, mids in matched.items() if mids - candidate.get(s1, set())\n        }\n        if offenders:\n            warnings.append(\n                f\"{len(offenders)} S1 entity(ies) have matched IDs not present in \"\n                f\"candidate_pairs.tsv, e.g. {examples(offenders)}. Final matches \"\n                \"normally come from your blocking candidates — double-check these.\"\n            )\n\n    return errors, warnings\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description=\"Validate ML Challenge 2026 submission output files before submitting.\"\n    )\n    parser.add_argument(\n        \"--matching\",\n        \"-m\",\n        default=\"output/matching_results.tsv\",\n        help=\"Path to matching_results.tsv (default: %(default)s)\",\n    )\n    parser.add_argument(\n        \"--candidate\",\n        \"-c\",\n        default=None,\n        help=\"Path to candidate_pairs.tsv \"\n        \"(default: output/candidate_pairs.tsv if it exists).\",\n    )\n    parser.add_argument(\n        \"--test-dir\",\n        \"-t\",\n        default=\"dataset/test\",\n        help=\"Folder with test_source1/2/3.tsv (default: %(default)s). \"\n        \"test_source2/3.tsv are only read when --check-ids is given.\",\n    )\n    parser.add_argument(\n        \"--check-ids\",\n        action=\"store_true\",\n        help=\"Also check that every matched/candidate ID exists in the test \"\n        \"Source-2/3 files. Off by default (loads all S2/S3 IDs into memory — a few \"\n        \"GB on the full test set). A nonexistent ID only lowers your score, so this \"\n        \"is a diagnostic, not a submission gate.\",\n    )\n    args = parser.parse_args()\n\n    # candidate_pairs.tsv is optional; default to the conventional path and let\n    # validate() skip (with a warning) if the file isn't there.\n    candidate_path = args.candidate or \"output/candidate_pairs.tsv\"\n\n    print(\"ML Challenge 2026 — submission validator\")\n    print(f\"  test dir: {args.test_dir}\")\n    try:\n        errors, warnings = validate(\n            args.matching, candidate_path, args.test_dir, check_ids=args.check_ids\n        )\n    except UnicodeDecodeError:\n        print()\n        print(\"FAIL — 1 issue(s) to fix before submitting:\")\n        print(\n            f\"  1. A file is not valid UTF-8 text (most likely {args.matching} or \"\n            f\"{candidate_path}). Re-save it as a plain UTF-8, tab-separated .tsv — \"\n            \"not cp1252/Latin-1, and not a compressed or binary file (.gz/.xlsx/\"\n            \".parquet) renamed to .tsv. In pandas: \"\n            \"df.to_csv(path, sep='\\\\t', index=False, encoding='utf-8').\"\n        )\n        return 1\n    except OSError as exc:\n        print()\n        print(\"FAIL — 1 issue(s) to fix before submitting:\")\n        print(f\"  1. Could not read a file: {exc}.\")\n        return 1\n\n    print()\n    for warning in warnings:\n        print(f\"WARNING: {warning}\")\n    if errors:\n        print(f\"FAIL — {len(errors)} issue(s) to fix before submitting:\")\n        for i, error in enumerate(errors, 1):\n            print(f\"  {i}. {error}\")\n        return 1\n    print(\"PASS — no blocking issues found. Safe to submit.\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n"}
for p, t in files.items():
    os.makedirs(os.path.dirname('/kaggle/working/er/'+p), exist_ok=True)
    open('/kaggle/working/er/'+p, 'w', encoding='utf-8').write(t)
print('wrote', len(files), 'files')


In [ ]:
# 2. install the two packages Kaggle may lack (needs Internet ON)
import importlib
missing = [m for m in ('rapidfuzz', 'unidecode') if importlib.util.find_spec(m) is None]
print('missing:', missing)
if missing:
    !pip install -q rapidfuzz unidecode
import rapidfuzz, unidecode
print('rapidfuzz', rapidfuzz.__version__, 'ok')

In [ ]:
# 3. locate data, set paths
import glob, os
tr = glob.glob('/kaggle/input/**/train/train_source1.tsv', recursive=True)
assert tr, 'attach the dataset kesavnimmagadda/amazon-ml-challenge via Add Input'
DATA = os.path.dirname(os.path.dirname(tr[0]))
CODE, WORK, OUT = '/kaggle/working/er', '/kaggle/working/work', '/kaggle/working/output'
os.chdir(CODE)
os.environ['PYTHONUTF8'] = '1'
print('DATA =', DATA, sorted(os.listdir(DATA)))

## Dry run (a few minutes): checks everything works before the long run

In [ ]:
!python -m src.pipeline train --data-dir {DATA} --work-dir /kaggle/working/work_dry --frac 0.01 --limit-s1 5000 --jobs 4
!python -m src.pipeline predict --data-dir {DATA} --work-dir /kaggle/working/work_dry --out /kaggle/working/output_dry --limit-s1 3000 --jobs 4

## Learn address aliases (optional; inspect the JSON afterwards and delete wrong entries)

In [ ]:
!python -m src.learn_aliases --data-dir {DATA} --out {WORK}/aliases.json

## Preprocess once (cached as parquet)

In [ ]:
!python -m src.pipeline preprocess --data-dir {DATA} --work-dir {WORK} --aliases {WORK}/aliases.json --jobs 4

## Train (smaller sample for Kaggle's 4 cores / ~30 GB). Check 'blocking recall' (>= 0.95) and the stage1 / stage2 held-out F0.5.
If it is too slow or runs out of memory, add `--k-rev 0 --k-addrc 0` (turns off the two extra retrieval views) or lower `--frac`.

In [ ]:
!python -m src.pipeline train --data-dir {DATA} --work-dir {WORK} --aliases {WORK}/aliases.json --frac 0.1 --jobs 4

## Predict the full test set

In [ ]:
!python -m src.pipeline predict --data-dir {DATA} --work-dir {WORK} --out {OUT} --jobs 4

## Validate, then download `output/matching_results.tsv` from the Output tab and upload to Unstop

In [ ]:
!python tools/validate_submission.py --matching {OUT}/matching_results.tsv --candidate {OUT}/candidate_pairs.tsv --test-dir {DATA}/test